<a href="https://colab.research.google.com/github/chrisstayte/DominoesCalculator/blob/dev/machine_learning/notebooks/05_v2_retraining_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Reviewed training code; rerun this cell after the one-time package restart.
from pathlib import Path
import json, sys, urllib.request
REVISION = "51f7014b428edf32df39f31b31555bfb098a271c"
ML_ROOT = Path("/content/domino-v2-reviewed")
FILES = ["scripts/run_model_stage.py", "notebooks/01_domino_detector.ipynb", "notebooks/02_pip_classifier.ipynb", "notebooks/04_test_photo.ipynb", "reviews/2026-11-22-photo-tests.json"]
for relative in FILES:
    target = ML_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/chrisstayte/DominoesCalculator/{REVISION}/machine_learning/{relative}", target)
sys.path.insert(0, str(ML_ROOT / "scripts"))
from run_model_stage import configured_cell
notebook = json.loads((ML_ROOT / "notebooks/02_pip_classifier.ipynb").read_text())
setup = next("".join(c["source"]) for c in notebook["cells"] if c["cell_type"] == "code" and "def install_packages(" in "".join(c["source"]))
print("Installing the reviewed training dependencies for", REVISION, flush=True)
exec(configured_cell(setup, {"ACTION": "train"}, "reviewed notebook setup"))
print("SETUP_READY: run the next cell to train both new models and compare the photos.")

Installing the reviewed training dependencies for 51f7014b428edf32df39f31b31555bfb098a271c
Installing packages for Python 3.13.16 — progress follows:
Installation finished. Continue to configuration.
SETUP_READY: run the next cell to train both new models and compare the photos.


In [3]:
# Mount the existing project, train sequentially, then compare all seven saved tests.
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import os, sys, json, subprocess
PROJECT = Path("/content/drive/MyDrive/DominoesCalculator")
JOB = PROJECT / "training_reports/v2-job"
JOB.mkdir(parents=True, exist_ok=True)
status = {"reviewed_commit": REVISION, "completed_stages": []}
for stage in ["classifier", "detector", "compare"]:
    status["current_stage"] = stage
    (JOB / "status.json").write_text(json.dumps(status, indent=2))
    print("\n========== STARTING " + stage.upper() + " ==========", flush=True)
    environment = dict(os.environ, MPLBACKEND="Agg", TF_CPP_MIN_LOG_LEVEL="2", PYTHONUNBUFFERED="1")
    with (JOB / (stage + ".log")).open("w") as log:
        process = subprocess.Popen([sys.executable, "-u", str(ML_ROOT / "scripts/run_model_stage.py"), stage], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=environment)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        code = process.wait()
    if code:
        status["failed_stage"] = stage
        (JOB / "status.json").write_text(json.dumps(status, indent=2))
        raise RuntimeError(f"{stage} stopped with exit code {code}. Saved log: {JOB / (stage + '.log')}")
    status["completed_stages"].append(stage)
status["current_stage"] = "complete"
(JOB / "status.json").write_text(json.dumps(status, indent=2))
comparison = json.loads((PROJECT / "training_reports/v2-comparison/completed.json").read_text())
print("\nALL TRAINING AND COMPARISON COMPLETE", flush=True)
print(json.dumps(comparison, indent=2), flush=True)

Mounted at /content/drive

========== STARTING CLASSIFIER ==========
classifier: code cell 1/15
classifier: code cell 2/15
classifier: code cell 3/15
classifier: code cell 4/15
classifier: code cell 5/15
Validated 36 photos and 821 domino boxes.
Persistent project: /content/drive/MyDrive/DominoesCalculator
Labeling/crops: /content/drive/MyDrive/DominoesCalculator/data/classifier/dominoes-v1/735d465ac37504ed-long-edge-rgb-128-v1
Model run: /content/drive/MyDrive/DominoesCalculator/models/classifier/pips-v2-01
classifier: code cell 6/15
Prepared 821 tiles / 1642 candidate halves.
Labeling status: {'labeled': 821}
classifier: code cell 7/15
Picker is paused. Set ACTION = "label", rerun Setup, then rerun this cell.
classifier: code cell 8/15
Labeling status: {'labeled': 821}
split  train  val  test
value                  
0         76   17    13
1         84   15    10
2         70   17    16
3         69   16    16
4         73   15    13
5         71   14    16
6         71   14    16
7 